<a href="https://colab.research.google.com/github/giacomomolinari/liar-fake-news-detector/blob/main/models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification Models

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import transformers
import random

from datasets import load_dataset
from google.colab import drive
from pathlib import Path

from imblearn.under_sampling import RandomUnderSampler


from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, accuracy_score

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from datasets import Dataset, load_dataset
from torch.utils.data import DataLoader

## 1. Getting the data

The notebook assumes that the LIAR dataset is available on your Google Drive at the path defined below. You can download the dataset from [William Yang Wang's website](https://sites.cs.ucsb.edu/~william/data/liar_dataset.zip).

In [ ]:
DATASET_PATH = Path("/content/drive/MyDrive/datasets/liar_dataset/")

In [ ]:
drive.mount('/content/drive')

In [ ]:
columns= ["ID", "Label", "Statement", "Subjects", "Speaker", "SpeakerJob", "State", "Party",
          "HistBarelyTrue", "HistFalse", "HistHalfTrue", "HistMostTrue", "HistPantsFire", "Context"]

In [ ]:
df_train = pd.read_csv(DATASET_PATH / "train.tsv", sep="\t", header=0, names=columns)
df_valid = pd.read_csv(DATASET_PATH / "valid.tsv", sep="\t", header=0, names=columns)
df_test = pd.read_csv(DATASET_PATH / "test.tsv", sep="\t", header=0, names=columns)

In [ ]:
df_train.head()

## 2. Preprocessing

### 2.1 Handling NaN values

As discussed in the eda notebook, only two columns have significant amounts of NaN values. I will remove these columns from the dataset to begin with.

In [ ]:
df_train.info()

In [ ]:
df_train_clean = df_train.drop(["SpeakerJob", "State"], axis=1)
df_train_clean.info()

For `NaN` values in the `Context` column, we will replace them with "no context provided."

In [ ]:
df_train_clean.loc[df_train_clean["Context"].isna(), "Context"] = "no context provided."

In [ ]:
df_train_clean[df_train_clean.isna().any(axis=1)]

Only two rows with NaN values remain, and these have all values NaN except for the statement. I will drop these for simplicity.

In [ ]:
df_train_clean = df_train_clean.dropna()

Let's define a convenience function to apply the same changes to the validation and test dataframes

In [ ]:
def clean_dataset(df):
  df_clean = df.drop(["SpeakerJob", "State"], axis=1)
  df_clean.loc[df_clean["Context"].isna(), "Context"] = "no context provided."
  df_clean = df_clean.dropna()
  return df_clean

In [ ]:
df_valid_clean = clean_dataset(df_valid)

df_valid_clean.info()

### 2.2 Selecting and Aggregating Features

As discussed in the EDA notebook, we will aggregate the `pants-fire` label with the `false` label to ensure the model is only learning about statement truthfulness, rather than learning possibly normative features like whether a lie is obvious, brazen, shocking or extravagant.

In [ ]:
df_train_reduced = df_train_clean.copy()

In [ ]:
df_train_reduced.loc[df_train_reduced["Label"] == "pants-fire", "Label"] = "false"

Then we will drop the "speaker history" features which contain the count of entries with each of the labels for the statement's speaker, and thus introduce data leak risks.

In [ ]:
df_train_reduced = df_train_reduced.drop(["HistBarelyTrue", "HistFalse", "HistHalfTrue", "HistMostTrue", "HistPantsFire"], axis=1)

In [ ]:
df_train_reduced.head()

Again we define a utility function to perform this transformation.

In [ ]:
def aggregate_and_drop(df):
  df_res = df.copy()
  df_res.loc[df_res["Label"] == "pants-fire", "Label"] = "false"
  df_res = df_res.drop(["HistBarelyTrue", "HistFalse", "HistHalfTrue", "HistMostTrue", "HistPantsFire"], axis=1)
  return df_res

In [ ]:
df_valid_reduced = aggregate_and_drop(df_valid_clean)

df_valid_reduced.head()

### 2.3 Multi-hot encoding of Subject feature

The Subject feature is a list of tags, so we will encode it by creating a new feature for each (sufficiently common) tag, which is 1 iff the sample in question has that subject tag.

In [ ]:
subject_set = set([])
subject_series = []
for subject in df_train_reduced["Subjects"]:
  if not isinstance(subject, str):
    print(subject)
  subject = subject.replace(", ", ",")
  split = subject.split(",")
  for s in split:
    if s not in subject_set:
      subject_set.add(s)
    subject_series.append(s)

In [ ]:
subject_series = pd.Series(subject_series)
counts = subject_series.value_counts()
counts.count(), counts[counts > 100].count()

In [ ]:
common_subjects = set(counts[counts > 100].index)

there are 100 subjects which appear over 40 times, while all others appear less than 40 times each. I will aggregate these infrequent subject tags into a new tag "other"

In [ ]:
df_train_listsubject = df_train_reduced.copy()
df_train_listsubject["Subjects"] = (df_train_listsubject["Subjects"].apply(lambda x: x.replace(", ", ",").split(","))
  .apply(lambda x: [s if s in common_subjects else "other" for s in x])
  .apply(lambda x: list(set(x)))) # remove duplicates

In [ ]:
df_train_listsubject.shape

In [ ]:
df_exploded = df_train_listsubject.explode("Subjects")
df_exploded.head()

In [ ]:
one_hot = pd.get_dummies(df_exploded['Subjects'])

In [ ]:
one_hot.shape

In [ ]:
multi_hot = one_hot.groupby(one_hot.index).sum()
multi_hot.shape

In [ ]:
df_train_multihot = df_train_reduced.merge(multi_hot, left_index=True, right_index=True).drop("Subjects", axis=1)
df_train_multihot.shape


To the original 7 features we added 101 subject features (100 for common subjects, 1 for "other"), and then removed the pre-existing "Subjects" feature, for a total of 107 features in the final dataset.

Again we define a utility function to more easily perform this transformation on the validation and test datasets.

In [ ]:
def multi_hot_encode(df, subject_set):
  df_listsubject = df.copy()
  df_listsubject["Subjects"] = (df_listsubject["Subjects"].apply(lambda x: x.replace(", ", ",").split(","))
    .apply(lambda x: [s if s in subject_set else "other" for s in x])
    .apply(lambda x: list(set(x)))) # remove duplicates

  df_exploded = df_listsubject.explode("Subjects") # add new rows for each value in "Subjects" list, copying other fields

  one_hot = pd.get_dummies(df_exploded['Subjects']) # one-hot encode all possible "Subject" values
  multi_hot = one_hot.groupby(one_hot.index).sum()  # group back by index to restore origina number of columns
  return multi_hot.merge(df_listsubject.drop("Subjects", axis=1), left_index=True, right_index=True)

In [ ]:
df_valid_reduced.shape

In [ ]:
df_valid_multihot = multi_hot_encode(df_valid_reduced, common_subjects)
df_valid_multihot.shape

### 2.4 Statement Length

I will remove all statements with over 40 words from the dataset. As shown in the EDA notebook, this adds up to about 100 statements, some of which are hundreds of words long, well above the dataset mode of 15. By removing these long statements we reduce the amount of padding needed to ensure all sequences processed by the RNN have the same length, which can improve performance.

In [ ]:
df_train_preprocessed = df_train_multihot.copy()
df_train_preprocessed["StatementLength"] = df_train_preprocessed["Statement"].apply(lambda x: len(x.split(" ")))
df_train_preprocessed[["Statement", "StatementLength"]].head()

In [ ]:
df_train_preprocessed.shape

In [ ]:
df_train_preprocessed = df_train_preprocessed[df_train_preprocessed["StatementLength"] < 40]
df_train_preprocessed.shape

As usual we define a utility function for this transformation.

In [ ]:
def drop_statements_longer_than(df, maxLength):
  df_res = df.copy()
  df_res["StatementLength"] = df_res["Statement"].apply(lambda x: len(x.split(" ")))
  df_res = df_res[df_res["StatementLength"] < maxLength]
  return df_res

In [ ]:
df_valid_preprocessed = drop_statements_longer_than(df_valid_multihot, 40)
df_valid_preprocessed.shape

## 3. Baseline Model 1: Metadata-only classifier

As a first baseline let's build a model that tries to classify statements based only on the available metadata: the name of the speaker, their political affiliation, and the topics touched on by the statement. I will also include the statement length, which is a derived piece of metadata and may potentially be helpful.

### 3.1 Preparing the data

We need a little bit more preprocessing before the metadata is ready to be used by a ML classifier. In particular, the remaining categorical features (`Speaker` and `Party`) must be converted to numerical ones, and the numerical feature `StatementLength` must be normalized.

#### 3.1.1 Normalizing StatementLength

In [ ]:
subjects_set_list = list(common_subjects.union({"other"}))
meta_features = ["Speaker", "Party", "StatementLength"] + subjects_set_list

X_train_meta = df_train_preprocessed[meta_features]
y_train_meta = df_train_preprocessed["Label"]

X_train_meta.head()

In [ ]:
min_max_scaler = MinMaxScaler(feature_range=(0, 1))
X_train_meta[["StatementLength"]] = min_max_scaler.fit_transform(X_train_meta[["StatementLength"]])
X_train_meta.head()

In [ ]:
statement_length_normalized = X_train_meta["StatementLength"]
statement_length_normalized.describe()

#### 3.1.2. Encoding Speaker

In [ ]:
speaker_counts = X_train_meta["Speaker"].value_counts()
speaker_counts

Clearly there are too many speakers for one-hot encoding. But for many of these speakers, they appear too infrequently for the model to actually be able to learn much about them. So let's look at how many speakers appear frequently.

In [ ]:
speaker_counts[speaker_counts>50].count()

Interestingly, only 65 speakers appear over 20 times. So we will aggregate all other speakers to a new value "other", and then one-hot encode this feature to generate 66 more feature (removing the existing `Speaker` feature)

In [ ]:
common_speakers = set(speaker_counts[speaker_counts>50].index)

X_train_speaker_encoded = X_train_meta.copy()
X_train_speaker_encoded["Speaker"] = X_train_meta["Speaker"].apply(lambda x: x if x in common_speakers else "other")
X_train_speaker_encoded = pd.get_dummies(X_train_speaker_encoded, columns=["Speaker"])
X_train_speaker_encoded.head()

#### 3.1.3 Encoding Party

We can one-hot encode the `Party` feature in a similar way.

In [ ]:
party_counts = X_train_meta["Party"].value_counts()
party_counts[party_counts>50]

In [ ]:
party_counts[party_counts>50]

Again it makes sense to aggregate parties that are too infrequent to be actually learned by the model.

In [ ]:
common_parties = set(party_counts[party_counts>50].index)
X_train_party_encoded = X_train_speaker_encoded.copy()

X_train_party_encoded["Party"] = X_train_speaker_encoded["Party"].apply(lambda x: x if x in common_parties else "other")
X_train_party_encoded = pd.get_dummies(X_train_party_encoded, columns=["Party"])
X_train_party_encoded.head()

In [ ]:
X_train_party_encoded.shape

Finally we convert the boolean values to integers for numerical processing and consistency.

In [ ]:
X_train_meta = X_train_party_encoded.copy()

X_statement_length = X_train_meta["StatementLength"].copy()
X_train_meta = X_train_meta.map(lambda x: int(x))
X_train_meta["StatementLength"] = X_statement_length

X_train_meta.head()

As before we define a function that performs all these transformations on a dataset.

In [ ]:
def meta_preprocessing(df):
  df_res = df.copy()

  min_max_scaler = MinMaxScaler(feature_range=(0, 1))
  df_res["StatementLength"] = min_max_scaler.fit_transform(df[["StatementLength"]])

  df_res["Speaker"] = df_res["Speaker"].apply(lambda x: x if x in common_speakers else "other")
  df_res = pd.get_dummies(df_res, columns=["Speaker"])

  df_res["Party"] = df_res["Party"].apply(lambda x: x if x in common_parties else "other")
  df_res = pd.get_dummies(df_res, columns=["Party"])

  df_statement_length = df_res["StatementLength"].copy()
  df_res = df_res.map(lambda x: int(x))
  df_res["StatementLength"] = df_statement_length

  return df_res

In [ ]:
X_valid_meta = df_valid_preprocessed[meta_features]
y_valid_meta = df_valid_preprocessed["Label"]

X_valid_meta.head()

In [ ]:
X_valid_meta = meta_preprocessing(X_valid_meta)
X_valid_meta.head()

This has one fewer column than our preprocessed training dataset, so there is probably a categorical feature that never appears in the validation set but which appears in the training set.

In [ ]:
X_valid_meta.columns.difference(X_train_meta.columns)
X_train_meta.columns.difference(X_valid_meta.columns)


It looks like a speaker that appears frequently in training data never appears in the validation data. We can fix this by just adding this column to the validation dataset and filling it with zeros

In [ ]:
X_valid_meta["Speaker_rush-limbaugh"] = 0
X_valid_meta.shape

In [ ]:
## Reorder validation dataframe so that features are in same order as training one
## (necessary for scikit learn's predictors)
X_valid_meta = X_valid_meta[X_train_meta.columns]
X_valid_meta.shape

### 3.2 Selecting and training models

#### 3.2.1 Defining the evaluation measures

We will evaluate our models by first converting the labels into integers and then calculating the MAE. This preserves the intuitive ordering that is intrinsic in these truthfulness labels. For example, if a statement is almost-true, it seems better to have classified it as true than to have classified it as false (although one could plausibly wish to specify just how much better that is, I will for simplicity just take this to be specified by the MAE metric).

I will also evaluate our model using the usual accuracy metric, for comparison and more interpretable results.

In [ ]:
label_to_int = {"false": 0, "barely-true": 1, "half-true": 2, "mostly-true": 3, "true": 4}

y_train_meta_int = y_train_meta.apply(lambda x: label_to_int[x])
y_valid_meta_int = y_valid_meta.apply(lambda x: label_to_int[x])

In [ ]:
scoring_rule = "accuracy"
scoring_rule_ordinal = "neg_mean_absolute_error"

#### 3.2.2 Simple Baseline Classifiers

As a sanity check, it's useful to define a **Majority Class Classifier**. This is a classifier which just predicts the most frequent class regardless of input. Its performance on the task provides a performance floor which our models should be able to clear if they have learned anything at all

In [ ]:
y_mode = y_train_meta_int.mode()[0]
y_mode

In [ ]:
def majority_class_classifier(X, y):
  return np.ones((len(X), 1))* y.mode()[0]

In [ ]:
mean_absolute_error(y_valid_meta_int, majority_class_classifier(X_valid_meta, y_train_meta_int))

In [ ]:
accuracy_score(y_valid_meta_int, majority_class_classifier(X_valid_meta, y_train_meta_int))

As expected this has an accuracy of around 30%, as around 27% of statements in the training dataset are false.

#### 3.2.3 Tree classifier

We use a simple tree classifier, limiting depth to avoid it catastrophically overfitting the training set.

In [ ]:
tree_model = DecisionTreeClassifier(random_state=42, max_depth=15)

tree_model

In [ ]:
tree_model_cv = -cross_val_score(tree_model, X_train_meta, y_train_meta_int, scoring=scoring_rule_ordinal, cv=10)

tree_model_cv.mean()

In [ ]:
tree_model.fit(X_train_meta, y_train_meta_int)

In [ ]:
mean_absolute_error(y_valid_meta_int, tree_model.predict(X_valid_meta))

In [ ]:
accuracy_score(y_valid_meta_int, tree_model.predict(X_valid_meta))

This does noticeably better than the majority class predictor in both MAE and accuracy, but it still has far from a great performance. To check that it's in fact not overfitting, let's look at the accuracy on training set

In [ ]:
accuracy_score(y_train_meta_int, tree_model.predict(X_train_meta))

Slightly better, but no sign of strong overfit.

In [ ]:
y_pred = tree_model.predict(X_valid_meta)
pd.Series(y_pred).value_counts()

We can see here that the model almost degenerates to the majority classifier. The model is not really learning from the features - rather it is learning that False is the most probable label, and hedging with some barely-true or half-true predictions.

#### 3.2.4 Random forest classifier

Let's try a slightly more powerful model, with slightly more fine-tuning.

In [ ]:
forest_model = RandomForestClassifier(random_state=42)

forest_model_cv = cross_val_score(forest_model, X_train_meta, y_train_meta_int, scoring=scoring_rule, cv=5)

forest_model_cv.mean()

In [ ]:
forest_model.fit(X_train_meta, y_train_meta_int)

In [ ]:
mean_absolute_error(y_valid_meta_int, forest_model.predict(X_valid_meta))

In [ ]:
accuracy_score(y_valid_meta_int, forest_model.predict(X_valid_meta))

In [ ]:
accuracy_score(y_train_meta_int, forest_model.predict(X_train_meta))

Much like for the tree classifier, the unconstrained version of RandomForest heavily overfits the training data. So let's look for some regularization hyperparameter values that improve this.

In [ ]:
parameter_grid = [
    {
        "max_depth": [10, 15, 20, 25, 30],
        "max_features": [10, 20, 30, 40, 50]
    }
]

In [ ]:
# forest_cls = RandomForestClassifier(random_state=42)

# grid_search = GridSearchCV(forest_cls, parameter_grid, cv=5, scoring=scoring_rule)
# grid_search.fit(X_train_meta, y_train_meta_int)
# grid_search.best_params_

In [ ]:
forest_model = RandomForestClassifier(random_state=42, max_depth=10, max_features=30)


forest_model.fit(X_train_meta, y_train_meta_int)

In [ ]:
mean_absolute_error(y_valid_meta_int, forest_model.predict(X_valid_meta))

In [ ]:
accuracy_score(y_valid_meta_int, forest_model.predict(X_valid_meta))

In [ ]:
accuracy_score(y_train_meta_int, forest_model.predict(X_train_meta))

In [ ]:
pd.Series(y_pred).value_counts()

Similar problem as the tree model, the RandomForest almost degenerates to a majority classifier!

#### 3.2.5 Reducing the feature space

In [ ]:
X_train_meta_reduced = X_train_meta.drop(common_subjects, axis=1)
X_train_meta_reduced = X_train_meta_reduced.drop(["Party_" + x for x in common_parties.union(set(['other']))], axis=1)
X_train_meta_reduced.head()

X_valid_meta_reduced = X_valid_meta.drop(common_subjects, axis=1)
X_valid_meta_reduced = X_valid_meta_reduced.drop(["Party_" + x for x in common_parties.union(set(['other']))], axis=1)
X_valid_meta_reduced.head()

In [ ]:
tree_model = DecisionTreeClassifier(random_state=42, max_depth=11)

tree_model_cv = -cross_val_score(tree_model, X_train_meta_reduced, y_train_meta_int, scoring=scoring_rule_ordinal, cv=10)

tree_model_cv.mean()

In [ ]:
tree_model.fit(X_train_meta_reduced, y_train_meta_int)

In [ ]:
mean_absolute_error(y_valid_meta_int, tree_model.predict(X_valid_meta_reduced)), accuracy_score(y_valid_meta_int, tree_model.predict(X_valid_meta_reduced))

In [ ]:
accuracy_score(y_train_meta_int, tree_model.predict(X_train_meta_reduced))

In [ ]:
y_pred = tree_model.predict(X_valid_meta_reduced)
pd.Series(y_pred).value_counts()

Still overfitting unless we let it deteriorate to majority classifier.

#### 3.2.6 Support Vector Classifier

Let's try using a support vector classifier, as SVCs tend to work well in high-dimensional spaces.

In [ ]:
svm_clf = SVC(random_state=42)

svm_clf_cv = -cross_val_score(svm_clf, X_train_meta, y_train_meta_int, scoring=scoring_rule_ordinal, cv=5)

svm_clf_cv.mean()

This is not much better than previous examples.

In [ ]:
svm_clf.fit(X_train_meta, y_train_meta_int)

In [ ]:
pd.Series([mean_absolute_error(y_valid_meta_int, svm_clf.predict(X_valid_meta)),
 accuracy_score(y_valid_meta_int, svm_clf.predict(X_valid_meta)),
 accuracy_score(y_train_meta_int, svm_clf.predict(X_train_meta))], index=["MAE", "AccuracyValid", "AccuracyTest"])

In [ ]:
pd.Series(svm_clf.predict(X_valid_meta)).value_counts()

We observe the same sort of deterioration to majority classifier.

#### 3.2.7 Rebalancing class frequencies

In [ ]:
UnderSampler = RandomUnderSampler(random_state=42, replacement=True)

X_under, y_under = UnderSampler.fit_resample(X_train_meta, y_train_meta_int)

X_under_valid, y_under_valid = UnderSampler.fit_resample(X_valid_meta, y_valid_meta_int)

y_under.value_counts(), y_under_valid.value_counts()

After undersampling all 5 classes have the exact same frequency in the dataset. Hence our baseline majority classifier will have 20% accuracy.

In [ ]:
accuracy_score(y_under_valid, majority_class_classifier(X_under_valid, y_under))

In [ ]:
X_under

In [ ]:
X_under_reduced = X_under.drop(common_subjects, axis=1)
X_under_reduced = X_under.drop(["Party_" + x for x in common_parties.union(set(['other']))], axis=1)
X_under_reduced.head()

X_under_valid_reduced = X_under_valid.drop(common_subjects, axis=1)
X_under_valid_reduced = X_under_valid.drop(["Party_" + x for x in common_parties.union(set(['other']))], axis=1)
X_under_valid_reduced.head()

In [ ]:
svm_clf = SVC(random_state=42, C=10)

svm_clf_cv = -cross_val_score(svm_clf, X_under_reduced, y_under, scoring=scoring_rule_ordinal, cv=5)

svm_clf_cv.mean()

In [ ]:
svm_clf.fit(X_under, y_under)

In [ ]:
pd.Series([mean_absolute_error(y_under, svm_clf.predict(X_under)),
 accuracy_score(y_under_valid, svm_clf.predict(X_under_valid)),
 accuracy_score(y_under, svm_clf.predict(X_under))], index=["MAE", "AccuracyValid", "AccuracyTest"])

In [ ]:
pd.Series(svm_clf.predict(X_under_valid)).value_counts()

In [ ]:
tree_model = DecisionTreeClassifier(random_state=42, max_depth=15)

tree_model_cv = -cross_val_score(tree_model, X_under, y_under, scoring=scoring_rule_ordinal, cv=10)

tree_model_cv.mean()

In [ ]:
tree_model.fit(X_under, y_under)

In [ ]:
pd.Series([mean_absolute_error(y_under, tree_model.predict(X_under)),
 accuracy_score(y_under_valid, tree_model.predict(X_under_valid)),
 accuracy_score(y_under, tree_model.predict(X_under))], index=["MAE", "AccuracyValid", "AccuracyTest"])

In [ ]:
forest_model = RandomForestClassifier(random_state=42, max_depth=10, n_estimators=1000)

forest_model.fit(X_under, y_under)

In [ ]:
forest_model_cv = -cross_val_score(forest_model, X_under, y_under, scoring=scoring_rule_ordinal, cv=10)

forest_model_cv.mean()

In [ ]:
pd.Series([mean_absolute_error(y_under, forest_model.predict(X_under)),
 accuracy_score(y_under_valid, forest_model.predict(X_under_valid)),
 accuracy_score(y_under, forest_model.predict(X_under))], index=["MAE", "AccuracyValid", "AccuracyTest"])

This model reaches over 27% in validation accuracy, which is actually noticeably better than the majority model now that all labels have been rebalanced by undersampling. After rebalancing, the baseline majority classifier only has 20% accuracy, so this is a 7% improvement. Furthermore, the accuracy achieved by this model is comparable to that achieved in [William Wang's paper](https://https://aclanthology.org/P17-2067/) which introduced the LIAR dataset, when evaluating models that use metadata only.

The model is still overfitting, but validation accuracy seems to decrease too when introducing more regularization.

In [ ]:
pd.Series(forest_model.predict(X_under_valid)).value_counts()

We can see that, while the model is slightly skewed towards predicting falsehood still, the distribution of predictions is far more balanced.

## 4. Baseline Model 2: Text-only classifier

### 4.1 Preparing the data

In [ ]:
statements_train = list(df_train_preprocessed["Statement"].apply(lambda x: x.lower()))
statements_valid = list(df_valid_preprocessed["Statement"].apply(lambda x: x.lower()))

statements_train[:3]

### 4.2 Tokenizer

I will use a pre-trained WordPiece tokenizer extracted from a BERT model.

In [ ]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
bert_encoding = bert_tokenizer(statements_train, padding=True, truncation=True, max_length=80, return_tensors="pt")


We can plot the length (in tokens) of each sequence by plotting the sum of the attention masks for each token.

In [ ]:
pd.Series([attention_mask.sum() for attention_mask in bert_encoding["attention_mask"]]).apply(lambda x: int(x)).plot.hist()

### 4.3 Building the Datasets and DataLoaders

First we convert our Pandas dataframes into Dataset objects

In [ ]:
label_to_int = {"false": 0, "barely-true": 1, "half-true": 2, "mostly-true": 3, "true": 4}


df_train_textonly = df_train_preprocessed[["Statement", "Label"]].copy()
df_train_textonly["Label"] = df_train_preprocessed["Label"].apply(lambda x: label_to_int[x])
df_train_textonly["Statement"] = df_train_preprocessed["Statement"].apply(lambda x: x.lower())


df_valid_textonly = df_valid_preprocessed[["Statement", "Label"]].copy()
df_valid_textonly["Label"] = df_valid_preprocessed["Label"].apply(lambda x: label_to_int[x])
df_valid_textonly["Statement"] = df_valid_preprocessed["Statement"].apply(lambda x: x.lower())

df_train_textonly.head()

In [ ]:
ds_train = Dataset.from_pandas(df_train_textonly)
ds_valid = Dataset.from_pandas(df_valid_textonly)

ds_train

Then we can use DataLoaders to tokenize the statements as they are passed to the model.

In [ ]:
def collate_fn(batch, tokenizer = bert_tokenizer):
  statements = [x["Statement"] for x in batch]
  labels = [[x["Label"]] for x in batch]

  encodings = tokenizer(statements, padding=True, truncation=True, max_length=80, return_tensors="pt")

  labels = torch.tensor(labels, dtype=torch.int64)

  return encodings, labels

In [ ]:
batch_size = 128 # increase to 256 if GPU allows it

text_train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
text_valid_loader = DataLoader(ds_valid, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

### 4.4 Building the Model

#### 4.4.1 Untrained Model (Training from scratch)

Let's define a class for our model

In [ ]:
class LieDetector_TextOnly(nn.Module):
  def __init__(self, vocab_size, n_layers=2, embed_dim = 128, hidden_dim=64, pad_id=0, dropout=0.2):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)

    self.output = nn.Linear(hidden_dim, 5) # output class logits

  def forward(self, encodings):
    embeddings = self.embedding(encodings["input_ids"])
    _outputs, hidden_states = self.gru(embeddings)
    return self.output(hidden_states[-1])

Then let's define training and evaluation functions, using the GPU for training.

In [ ]:
if torch.cuda.is_available():
    device="cuda"
elif torch.backends.mps.is_available:
    device="mps"
else:
    device="cpu"

device

In [ ]:
# Evaluation function (not using torchmetrics)
def evaluate(model, data_loader, criterion):
  model.eval()
  total_loss = 0
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)

      y_batch = y_batch.squeeze(1)

      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
  return total_loss/len(data_loader)

In [ ]:
# Evaluation function (not using torchmetrics)
def evaluate_as_regression(model, data_loader, criterion):
  model.eval()
  total_loss = 0
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch).softmax(dim=-1).float().to(device)

      y_batch = target_idx_to_vector(y_batch, 5)
      y_batch = y_batch.squeeze(1).to(device)

      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
  return total_loss/len(data_loader)

In [ ]:
# General training function
def train(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
  for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      y_batch = y_batch.squeeze(1)

      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()

      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    mean_loss = total_loss / len(train_loader)
    eval_loss = evaluate(model, valid_loader, criterion)
    print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {mean_loss:.4f}, Validation Loss: {eval_loss:.4f}")

In [ ]:
def target_idx_to_vector(target_idxs, length):
  target_vector = torch.zeros(len(target_idxs), length)#
  for i, target_idx in enumerate(target_idxs):
    target_vector[i, target_idx] = 1
  return target_vector

In [ ]:
# Train by converting the logits to probabilities and the targets to indicator vectors, then measuring divergence
# More computationally expensive than training with logits only
def train_as_regression(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
  for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch).softmax(dim=-1).float().to(device)
      y_batch = target_idx_to_vector(y_batch, 5)
      y_batch = y_batch.squeeze(1).to(device)

      """ DEBUG:
      print("Shapes:")
      print("y_pred: ", y_pred.shape)
      print("y_batch: ", y_batch.shape)

      print("Types:")
      print("y_pred: ", y_pred.dtype)
      print("y_batch: ", y_batch.dtype)

      print("Tensors:")
      print("y_pred: ", y_pred)
      print("y_batch: ", y_batch)

      print(loss.requires_grad)

      END DEBUG"""

      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()

      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    mean_loss = total_loss / len(train_loader)
    eval_loss = evaluate_as_regression(model, valid_loader, criterion)
    print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {mean_loss:.4f}, Validation Loss: {eval_loss:.4f}")

In [ ]:
torch.manual_seed(42)

In [ ]:
model = LieDetector_TextOnly(vocab_size=bert_tokenizer.vocab_size, pad_id=bert_tokenizer.pad_token_id).to(device)


We compute class frequencies on the non-rebalanced dataset to use the weights in scoring.

In [ ]:
class_counts = np.array(df_train_textonly.groupby("Label").count().values.flatten())
class_counts

In [ ]:
class_weights = 1.0/class_counts                                  # weight inversely proportional to number of occurrences
class_weights = torch.tensor(class_weights/ class_weights.sum())  # normalize

class_weights = class_weights.float().to(device)
class_weights

In [ ]:
learning_rate = 0.0005
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
train(model,  optimizer=optimizer, criterion=criterion, train_loader = text_train_loader, valid_loader = text_valid_loader, n_epochs=25)

The model seems to be overfitting to an extreme degree, where the validation performance actually gets worse and worse as the model fits to the training dataset.



Let's write a simple utility function to give us an intuitive sense of how the model performs on a few validation statements

In [ ]:
def sample_validation_results(model, validation_set, sample_size):
  valid_length = len(validation_set)
  statement_list = list(validation_set["Statement"])
  label_list = list(validation_set["Label"])
  sample = random.sample(range(0, valid_length), sample_size)

  df = pd.DataFrame(columns=["Statement", "Label", "Prediction"])

  for entry_idx in sample:
    statement = statement_list[entry_idx]
    label = label_list[entry_idx]

    tokenized_statement = bert_tokenizer(statement, padding=True, truncation=True, max_length=80, return_tensors="pt")

    model.eval()
    with torch.no_grad():
      prediction = model(tokenized_statement.to(device)).argmax(dim=-1).item()
      df.loc[len(df)] = [statement, label, prediction]
  return df



In [ ]:
sample_validation_results(model, ds_valid, 10)

This clearly shows that the model is performing very poorly. The likely reason for this poor performance is that the model just doesn't have enough information available in the dataset to learn reasonably informative/meaningful embeddings of the tokens, which in turn are necessary for determining whether the statements are true or false. A better approach would be to reuse the pre-trained embedding layer of an existing model, which can represent tokens into reasonably informative embeddings, and then focus our model training on determining degree of truthfulness from these embeddings.

## 5. Main Model: Text and metadata classifier